In [ ]:
# Define the training logic for infomax transformers
import sys
from pathlib import Path

# Add pytutorials the path (one level up from notebooks/) to sys.path
sys.path.append(str(Path.cwd().parent))

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import numpy as np
from collections import defaultdict
from pathlib import Path

from pytutorials.data.synthetic_triplets import SyntheticDisentangleDataset, VOCAB_SIZE, decode_triplets, decode_target_color
from pytutorials.models.infomax import BaselineTransformer, InfoMaxTransformer
from pytutorials.training.infomax import train_model, eval_model
from pytutorials.utils.gpu import get_device, optimize_model
from pytutorials.utils.visualize import compute_entropy_divergence, visualize_attention_heads, plot_metrics


In [ ]:
# Check for available device
device = get_device()
print("Using", device)

In [ ]:
# ---- Main Training Routine ----
def run_experiment(d_model=256, nhead=4, num_layers=3, ff_dim=1024, dropout=0.1, num_epochs=30, device=None):
    
    train_loader = DataLoader(SyntheticDisentangleDataset(5000), batch_size=64, shuffle=True)
    val_loader   = DataLoader(SyntheticDisentangleDataset(1000), batch_size=64)

    baseline     = BaselineTransformer(VOCAB_SIZE, d_model, nhead, num_layers, ff_dim, dropout)
    baseline     = optimize_model(baseline, device=device, dtype=None, compile_model=False)
    
    infomax      = InfoMaxTransformer(VOCAB_SIZE, d_model, nhead, num_layers, ff_dim, dropout)
    infomax      = optimize_model(infomax, device=device, dtype=None, compile_model=False)

    opt_base     = torch.optim.Adam(baseline.parameters(), lr=1e-4)
    opt_info     = torch.optim.Adam(infomax.parameters(), lr=1e-4)
    criterion    = nn.CrossEntropyLoss()

    history_base = defaultdict(list)
    history_info = defaultdict(list)
    SAVE_PATH    = Path("./attention_maps")
    SAVE_PATH.mkdir(exist_ok=True)

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")

        tr_base  = train_model(baseline, opt_base, criterion, train_loader, model_name="baseline", cur_epoch=epoch, num_epochs=num_epochs, device=device)
        val_base = eval_model(baseline, criterion, val_loader, model_name="baseline", cur_epoch=epoch, num_epochs=num_epochs, device=device)
        
        for k, v in tr_base.items(): history_base[f"train_{k}"].append(v)
        for k, v in val_base.items(): history_base[f"val_{k}"].append(v)

        tr_info  = train_model(infomax, opt_info, criterion, train_loader, model_name="infomax", cur_epoch=epoch, num_epochs=num_epochs, device=device)
        val_info = eval_model(infomax, criterion, val_loader, model_name="infomax", cur_epoch=epoch, num_epochs=num_epochs, device=device)
        
        for k, v in tr_info.items(): history_info[f"train_{k}"].append(v)
        for k, v in val_info.items(): history_info[f"val_{k}"].append(v)

        print("[BASELINE] Loss: {:.4f}  Acc: {:.2%}  Ortho: {:.6f}  Entropy: {:.3f} ± {:.3f}".format(
            val_base['loss'], val_base['acc'], val_base['ortho'], val_base['entropy_mean'], val_base['entropy_std']))
        print("[INFOMAX ] Loss: {:.4f}  Acc: {:.2%}  Ortho: {:.6f}  Entropy: {:.3f} ± {:.3f}".format(
            val_info['loss'], val_info['acc'], val_info['ortho'], val_info['entropy_mean'], val_info['entropy_std']))

        # Qualitative Sample
        sample, target = next(iter(val_loader))
        sample, target = sample.to(device), target.to(device)

        print("RAW input:", sample[0].tolist())
        print("TRIPLETS:", decode_triplets(sample[0].tolist()))

        base_logits = baseline(sample)[0]
        info_logits = infomax(sample)[0]
        
        base_pred   = base_logits[:, 0::3, :].argmax(-1)[0].tolist()
        info_pred   = info_logits[:, 0::3, :].argmax(-1)[0].tolist()

        print(" target  :", decode_target_color(target[0].tolist()))
        print(" baseline:", decode_target_color(base_pred))
        print(" infomax :", decode_target_color(info_pred))

        # Visualization
        _, baseline_attn = baseline(sample)
        visualize_attention_heads(baseline_attn[-1], sample_idx=0, layer_idx=num_layers - 1, save_path=SAVE_PATH/f"baseline_attn_epoch{epoch+1}.png")
        
        _, infomax_attn, _ = infomax(sample)
        visualize_attention_heads(infomax_attn[-1], sample_idx=0, layer_idx=num_layers - 1, save_path=SAVE_PATH/f"infomax_attn_epoch{epoch+1}.png")

        if epoch == num_epochs - 1:
            plot_metrics(history_base, title="Baseline Transformer", save_path=SAVE_PATH/f"baseline_epoch{epoch+1}.png")
            plot_metrics(history_info, title="InfoMax Transformer", save_path=SAVE_PATH/f"infomax_epoch{epoch+1}.png")

    return history_base, history_info


In [ ]:
# Run training
if __name__ == '__main__':
    run_experiment(device=device)
